# marag-precision — Kaggle launcher

Thin wrapper around `src/runner.py`. **No logic lives here** — if you find yourself
editing pipeline code in this notebook, put it in the repo instead (SPEC §9).

## Before you run anything

| Setting | Value | Why |
|---|---|---|
| **Internet** | **On** (Settings → Internet) | HuggingFace model + dataset downloads fail silently without it |
| **Accelerator** | **GPU T4 x2** | Only one GPU is used; the second idles but the quota is identical |
| Model cache | `/kaggle/tmp` (set below) | `/kaggle/working` holds only ~20 GB and must not fill with weights |

Results are appended to `results/<run_id>.jsonl` **as each batch completes**, not at
the end. A session that dies at hour 11 loses nothing — rerun the same command and
it resumes (SPEC §6). Download the zip from cell 5 before the session expires.

Run IDs (SPEC §2): `baseline`, `planner_4bit`, `stepdef_4bit`, `extractor_4bit`, `qa_4bit`.

In [ ]:
# Pinned to the versions the pipeline was developed and validated against.
# transformers 5.x is required: models.py uses the  argument, which
# replaced  in v5. Unpinned installs have broken this before.
!pip install -q -U "transformers==5.14.1" "bitsandbytes==0.50.0" "datasets==5.0.1" accelerate pyyaml


In [ ]:
import os

# Keep weights off /kaggle/working (~20 GB budget, and it is what gets zipped).
os.environ['HF_HOME'] = '/kaggle/tmp/hf'
os.makedirs('/kaggle/tmp/hf', exist_ok=True)

REPO_URL = 'https://github.com/Retixx/Maxim-Mohareb-Michael-Zhang-Fun-Time.git'
REPO_DIR = '/kaggle/working/marag-precision'

if os.path.isdir(REPO_DIR):
    !cd {REPO_DIR} && git pull --ff-only
else:
    !git clone -q {REPO_URL} {REPO_DIR}

%cd {REPO_DIR}
!git log --oneline -1

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,memory.free,driver_version --format=csv

import torch
print('torch', torch.__version__, '| cuda', torch.cuda.is_available())
assert torch.cuda.is_available(), 'No GPU — set Accelerator to GPU T4 x2'
print(torch.cuda.get_device_name(0))

### Verify on 10 questions before any full run

SPEC §11 step 7. Do not launch a 300-question sweep until this cell has completed
end to end. `--seed 1234` is the development split; the real runs use the config's
`eval_seed`, which prompt work has never seen.

In [ ]:
!python smoke_test.py --run baseline --n 10 --seed 1234

### Persisting results (do this once)

Kaggle wipes `/kaggle/working` when the session ends, so an interactive run that
finishes and then goes idle loses everything. The cell below pushes `results/` to
the GitHub repo **after every run**, so nothing is ever lost.

It also makes sessions resumable: cell 2 clones the repo, which now brings the
already-completed results back down with it, and the runner skips them.

**One-time setup:**
1. GitHub -> Settings -> Developer settings -> Personal access tokens -> fine-grained
2. Repository access: only `Maxim-Mohareb-Michael-Zhang-Fun-Time`; permission **Contents: Read and write**
3. Kaggle notebook -> **Add-ons -> Secrets** -> add it as `GITHUB_TOKEN`, and attach it to this notebook

Without the secret the sweep still runs, it just won't push (you'll see a warning).


In [ ]:
import subprocess
from pathlib import Path

def _git(*args):
    return subprocess.run(['git', *args], cwd=REPO_DIR, capture_output=True, text=True)

PUSH_OK = False
try:
    from kaggle_secrets import UserSecretsClient
    _tok = UserSecretsClient().get_secret('GITHUB_TOKEN')
    # Written to a credential file, never onto a command line, so the token
    # cannot appear in cell output or in a saved notebook version.
    Path.home().joinpath('.git-credentials').write_text(
        'https://' + _tok + ':x-oauth-basic@github.com\n')
    del _tok
    _git('config', '--global', 'credential.helper', 'store')
    _git('config', 'user.email', 'kaggle-runner@local')
    _git('config', 'user.name', 'kaggle-runner')
    PUSH_OK = True
    print('GITHUB_TOKEN found - results will be pushed after each run')
except Exception as e:
    print('NO GITHUB_TOKEN secret (' + type(e).__name__ + ').')
    print('Sweep will still run, but results will NOT survive the session.')

def push_results(tag):
    """Commit and push results/ . Never raises - a push failure must not kill a sweep."""
    if not PUSH_OK:
        print('  [push] skipped (no token)')
        return
    try:
        # -f because results/ is gitignored for local development.
        _git('add', '-f', 'results/')
        c = _git('commit', '-m', 'results: ' + tag + ' (kaggle n=' + str(N) + ')')
        if c.returncode != 0:
            blob = (c.stdout + c.stderr).lower()
            if 'nothing to commit' in blob or 'no changes added' in blob:
                print('  [push] ' + tag + ': nothing new to commit')
            else:
                print('  [push] ' + tag + ': commit failed - ' + blob.strip()[-200:])
            return
        _git('pull', '--rebase', '--autostash', 'origin', 'main')
        p = _git('push', 'origin', 'HEAD:main')
        ok = p.returncode == 0
        print('  [push] ' + tag + ': ' + ('OK' if ok else 'FAILED ' + p.stderr.strip()[-300:]))
    except Exception as e:
        print('  [push] ' + tag + ' errored: ' + type(e).__name__ + ' ' + str(e)[:200])


In [ ]:
# Llama-3.2-3B is a GATED HuggingFace repo. Without an accepted licence and a
# token this fails at model load, several minutes in, not at clone time.
#   1. huggingface.co/meta-llama/Llama-3.2-3B-Instruct -> accept the licence
#   2. HF Settings -> Access Tokens -> create a READ token
#   3. Kaggle Add-ons -> Secrets -> add as HF_TOKEN, attach to this notebook
# Model 1 (Qwen) is ungated and needs none of this.
try:
    from kaggle_secrets import UserSecretsClient
    os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
    print('HF_TOKEN found - gated repos available')
except Exception as e:
    print('no HF_TOKEN (' + type(e).__name__ + ') - ungated models only')

# Fail fast rather than 20 minutes into a sweep.
MODEL_2 = 'meta-llama/Llama-3.2-3B-Instruct'
from huggingface_hub import model_info
try:
    model_info(MODEL_2, token=os.environ.get('HF_TOKEN'))
    print('access to ' + MODEL_2 + ': OK')
except Exception as e:
    print('CANNOT ACCESS ' + MODEL_2)
    print('  ' + type(e).__name__ + ': ' + str(e)[:200])
    print('  -> accept the licence and attach HF_TOKEN before running the model-2 sweep')


In [ ]:
# The full 4-bit tier: baseline + the four single-agent runs (SPEC section 2).
# Resume is default, so re-running this cell after a session dies costs nothing:
# completed agent calls are skipped. Results are pushed to GitHub after EACH run,
# so a session that dies loses at most the run in flight.
N = 750
RUN_IDS = ["baseline", "planner_4bit", "stepdef_4bit", "extractor_4bit", "qa_4bit"]

for rid in RUN_IDS:
    print("=" * 70, flush=True)
    print(">>> " + rid, flush=True)
    print("=" * 70, flush=True)
    !python -m src.runner --config config/experiment.yaml --run {rid}
    push_results(rid)


### Model 2 — Llama-3.2-3B-Instruct (SPEC §11 build step 9)

The generalization claim: does the role ordering hold on a different family and
tokenizer at the same capability tier? SPEC §2 rates this above extra bit-widths.

The model is a **config value, not a plugin** (SPEC §12) — `--model-id` overrides
`config/experiment.yaml`, nothing else changes. Results carry the model in their
filename, so Llama runs cannot collide with the Qwen ones.

**Budget:** ~6.4 GB at FP16 vs Qwen's 3.1 GB, so expect roughly 2x the wall time —
**~8-9 GPU-h for the five runs**, against a 12 h session cap and 30 GPU-h/week.
It fits in one session but with little margin. If it dies, results pushed after
each run mean you resume rather than restart.

Batch 16 may OOM where Qwen didn't (6.4 GB weights + activations on a 15 GB T4).
The autotune halves and retries — watch for `[oom] reducing batch size`, which
would be its first real firing.


In [ ]:
# Model 2 sweep. Same five runs, same n and seed, different base model.
# Self-contained: does not depend on the model-1 cell having been run.
MODEL_2 = 'meta-llama/Llama-3.2-3B-Instruct'
N = 750
RUN_IDS = ["baseline", "planner_4bit", "stepdef_4bit", "extractor_4bit", "qa_4bit"]

for rid in RUN_IDS:
    print('=' * 70, flush=True)
    print('>>> ' + rid + '  [' + MODEL_2 + ']', flush=True)
    print('=' * 70, flush=True)
    !python -m src.runner --config config/experiment.yaml --run {rid} --model-id {MODEL_2}
    push_results(rid + '-llama')


In [ ]:
# Zip results for download. Safe to run repeatedly, including mid-sweep.
!cd {REPO_DIR} && zip -qr /kaggle/working/results.zip results/
!ls -lh /kaggle/working/results.zip
!ls -l {REPO_DIR}/results/